# 🎙️ BeyondEcho — Voice Cloning Server

Runs **XTTS v2** (free, open-source) on a Colab T4 GPU and exposes it
over a public **ngrok** URL so your local / deployed BeyondEcho app can
call it for voice cloning and TTS.

**Before you start:**
1. Set Runtime → Change runtime type → **T4 GPU**
2. Get a free ngrok token at [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken)
3. Run cells top-to-bottom (Ctrl+F9)

---
### Why XTTS v2?
| Model | Quality | Colab T4 | Sample needed | License |
|-------|---------|----------|---------------|---------|
| **XTTS v2** (recommended) | ⭐⭐⭐⭐ | ✅ ~4 GB VRAM | 5–30 s | CPML (free non-commercial) |
| OpenVoice v2 | ⭐⭐⭐⭐ | ✅ ~2 GB | 3–10 s | MIT |
| F5-TTS | ⭐⭐⭐⭐½ | ✅ ~3 GB | 3–10 s | MIT |
| Bark | ⭐⭐⭐ | ✅ slow | prompt-based | MIT |

We use **XTTS v2** — best quality + already integrated in BeyondEcho's backend.

In [ ]:
# ── Cell 1: Check GPU & Install ─────────────────────────────────────────────
import subprocess, sys

# Confirm GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print(f'✅ GPU detected: {result.stdout.strip()}')
else:
    print('⚠️  No GPU detected — go to Runtime → Change runtime type → T4 GPU')

print('\n📦 Installing dependencies (takes ~2 min)...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'TTS==0.22.0',
    'fastapi',
    'uvicorn[standard]',
    'pyngrok',
    'python-multipart',
    'nest_asyncio',
], check=True)
print('✅ All packages installed')

In [ ]:
# ── Cell 2: Load XTTS v2 Model ───────────────────────────────────────────────
# First run downloads ~1.8 GB — subsequent runs load from cache.
import os, torch
from TTS.api import TTS

os.environ['COQUI_TOS_AGREED'] = '1'   # accept Coqui model terms

USE_GPU = torch.cuda.is_available()
print(f'Loading XTTS v2 on {"GPU" if USE_GPU else "CPU"} ...')

tts_model = TTS(
    model_name='tts_models/multilingual/multi-dataset/xtts_v2',
    gpu=USE_GPU,
)
print('✅ XTTS v2 ready!')

In [ ]:
# ── Cell 3: Define the FastAPI Voice Server ───────────────────────────────────
import uuid, io, os, torch
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import Response
from fastapi.middleware.cors import CORSMiddleware

SAMPLES_DIR = '/tmp/be_voice_samples'
os.makedirs(SAMPLES_DIR, exist_ok=True)

app = FastAPI(title='BeyondEcho Voice Server', version='1.0.0')

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_methods=['*'],
    allow_headers=['*'],
)


@app.get('/health')
async def health():
    return {
        'status': 'ok',
        'model': 'xtts_v2',
        'gpu': torch.cuda.is_available(),
    }


@app.post('/clone')
async def clone_voice(
    file: UploadFile = File(..., description='Audio sample (wav/mp3/webm, 5-30 sec)'),
    name: str = Form(default='voice'),
):
    """Save a speaker audio sample and return a voice_id."""
    audio_bytes = await file.read()
    if len(audio_bytes) < 1000:
        raise HTTPException(400, 'Audio file too small — record at least 5 seconds')

    voice_id = uuid.uuid4().hex
    # Save as .wav — ffmpeg handles format conversion automatically in TTS
    ext = (file.filename or 'sample.webm').rsplit('.', 1)[-1]
    sample_path = f'{SAMPLES_DIR}/{voice_id}.{ext}'
    with open(sample_path, 'wb') as f:
        f.write(audio_bytes)

    return {
        'voice_id': voice_id,
        'size_kb': len(audio_bytes) // 1024,
        'message': 'Voice sample saved successfully',
    }


@app.post('/synthesize')
async def synthesize(
    text: str = Form(...),
    voice_id: str = Form(...),
    language: str = Form(default='en'),
):
    """Convert text to speech using the cloned voice."""
    # Find the sample (any extension)
    sample_path = None
    for f in os.listdir(SAMPLES_DIR):
        if f.startswith(voice_id):
            sample_path = f'{SAMPLES_DIR}/{f}'
            break

    if not sample_path:
        raise HTTPException(404, f'voice_id not found: {voice_id}')

    buf = io.BytesIO()
    tts_model.tts_to_file(
        text=text,
        speaker_wav=sample_path,
        language=language,
        file_path=buf,
    )
    buf.seek(0)
    return Response(content=buf.read(), media_type='audio/wav')


@app.delete('/voice/{voice_id}')
async def delete_voice(voice_id: str):
    for f in os.listdir(SAMPLES_DIR):
        if f.startswith(voice_id):
            os.remove(f'{SAMPLES_DIR}/{f}')
    return {'status': 'deleted'}


print('✅ FastAPI routes defined')

In [ ]:
# ── Cell 4: Set your ngrok token ─────────────────────────────────────────────
# Get your FREE token at: https://dashboard.ngrok.com/get-started/your-authtoken
# (sign up takes 30 sec)

NGROK_TOKEN = ''   # ← paste token here

# Alternative: store in Colab Secrets (left sidebar 🔑) as NGROK_TOKEN
if not NGROK_TOKEN:
    try:
        from google.colab import userdata
        NGROK_TOKEN = userdata.get('NGROK_TOKEN')
        print('✅ Token loaded from Colab Secrets')
    except Exception:
        raise ValueError(
            'Set NGROK_TOKEN above or add it to Colab Secrets (🔑 icon in the sidebar)'
        )

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
print('✅ ngrok configured')

In [ ]:
# ── Cell 5: Launch! 🚀 ───────────────────────────────────────────────────────
import nest_asyncio
import threading
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

PORT = 7000

# Kill stale tunnels from previous runs
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

# Open new tunnel
tunnel = ngrok.connect(PORT, proto='http')
PUBLIC_URL = tunnel.public_url

print('=' * 62)
print('🎙️   BeyondEcho Voice Server is LIVE')
print('=' * 62)
print(f'''
📡  Public URL:  {PUBLIC_URL}

Copy these into your BeyondEcho .env:

  VOICE_PROVIDER=coqui
  COQUI_API_URL={PUBLIC_URL}

If deployed on Koyeb/Render, update the env var there too.

⚠️  Keep this tab open — closing it stops the server.
   The URL changes every restart (free ngrok tier).
''')
print('=' * 62)

# Run uvicorn in background thread
def _run():
    uvicorn.run(app, host='0.0.0.0', port=PORT, log_level='warning')

thread = threading.Thread(target=_run, daemon=True)
thread.start()

# Quick self-test
import time, requests
time.sleep(2)
try:
    r = requests.get(f'http://localhost:{PORT}/health')
    print(f'Health check: {r.json()}')
except Exception as e:
    print(f'Health check failed: {e}')

---
## 🧪 Test the API (optional)

Run the cell below to do a quick voice clone + synthesis test using a sample WAV.
This confirms everything works before you connect BeyondEcho.

In [ ]:
# ── Cell 6 (optional): Quick test ─────────────────────────────────────────────
import requests
from IPython.display import Audio, display

BASE = f'http://localhost:{PORT}'

# Generate a tiny test WAV using TTS itself (or upload your own below)
import io, scipy.io.wavfile as wav
import numpy as np

# 2-second sine tone as dummy speaker sample (replace with real voice for quality)
sr = 22050
t  = np.linspace(0, 2, sr * 2)
tone = (np.sin(2 * np.pi * 220 * t) * 32767).astype(np.int16)
buf = io.BytesIO()
wav.write(buf, sr, tone)
buf.seek(0)

# 1. Clone
clone_r = requests.post(f'{BASE}/clone', files={'file': ('test.wav', buf, 'audio/wav')},
                         data={'name': 'test'})
voice_id = clone_r.json()['voice_id']
print(f'✅ Cloned voice_id: {voice_id}')

# 2. Synthesize
synth_r = requests.post(f'{BASE}/synthesize',
                         data={'text': 'Hello from BeyondEcho voice server!', 'voice_id': voice_id})
print(f'✅ Synthesized {len(synth_r.content):,} bytes of audio')
display(Audio(synth_r.content, rate=24000))

# 3. Clean up
requests.delete(f'{BASE}/voice/{voice_id}')
print('✅ Test complete — real voice cloning requires a proper audio sample')